# LLM-G-IDS UNSW-NB15 Instructor Pipeline

This notebook runs the same production pipeline used by the repository, phase by phase, on UNSW-NB15. It is designed for Google Colab with Google Drive checkpoints so the instructor can run a phase, stop, and resume later.

Expected pooled out-of-fold macro-F1 ladder:

```text
GNN 0.5496 < LLM 0.7353 < AGAF 0.7459 < Feedback loop 0.7764
```

The feedback loop uses the prototype-only semantic consultant, `TOP_K_PERCENT = 16.0`, and `BIAS_CONFIDENCE_FRAC = 0.5`. The previous entropy and semantic-confidence sweeps are documented research decisions; they are not rerun in this notebook.

Important limitation: the pipeline uses label-conditioned edge aggregation in Step 1, so the results are research cross-validation estimates rather than deployment-valid estimates. The feedback-vs-AGAF confidence interval crosses zero, so the ladder is an observed point-estimate result, not statistical proof of universal dominance.


## Phase Overview

1. Mount Google Drive and configure deterministic CPU execution.
2. Validate the shared Drive raw CSV files: `UNSW-NB15_1.csv` through `UNSW-NB15_4.csv`.
3. Preprocess raw flows into the normalized ToN-style CSV.
4. Build the graph and stratified edge folds.
5. Build KG triples and label-free natural-language triples.
6. Build honest GNN out-of-fold logits and out-of-fold GNN embeddings.
7. Encode KG text with CySecBERT and build whitened prototypes.
8. Train AGAF using only `edge_embeddings_oof.pt`.
9. Train the prototype-only feedback loop and assemble the final ladder.


In [1]:
# PHASE: setup
# Colab/bootstrap cell. Run this first.
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/alialzein01/LLM-G-IDS.git'
# Pin this to the committed branch/tag you share with the instructor.
REPO_REF = 'feature/instructor-colab-notebook'
if IN_COLAB:
    REPO_DIR = Path('/content/LLM-G-IDS')
else:
    candidate_root = Path.cwd().resolve()
    if not (candidate_root / 'src').exists() and (candidate_root.parent / 'src').exists():
        candidate_root = candidate_root.parent
    REPO_DIR = candidate_root
DRIVE_ROOT = Path('/content/drive/MyDrive/LLM-G-IDS-Instructor') if IN_COLAB else REPO_DIR / 'drive_instructor_demo'
RAW_DRIVE_DIR = DRIVE_ROOT / 'raw'
CHECKPOINT_ROOT = DRIVE_ROOT / 'checkpoints'
RESTORE_IF_AVAILABLE = True
FORCE_REBUILD = False
SEED = 42
PYTORCH_VERSION = '2.4.1'
PYG_WHEEL_INDEX = 'https://data.pyg.org/whl/torch-2.4.1+cpu.html'
CYSECBERT_MODEL_REVISION = 'main'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Repository root: {Path.cwd()}')
print(f'Drive root: {DRIVE_ROOT}')
print(f'Checkpoint root: {CHECKPOINT_ROOT}')


Repository root: /Users/ali/Desktop/ids-framework-instructor-notebook
Drive root: /Users/ali/Desktop/ids-framework-instructor-notebook/drive_instructor_demo
Checkpoint root: /Users/ali/Desktop/ids-framework-instructor-notebook/drive_instructor_demo/checkpoints


In [2]:
# PHASE: setup
# Install dependencies in Colab. Local Jupyter users can skip if requirements are already installed.
if IN_COLAB:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        f'torch=={PYTORCH_VERSION}',
        '--index-url', 'https://download.pytorch.org/whl/cpu',
    ], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        '-r', 'notebooks/colab-requirements.txt',
        '-f', PYG_WHEEL_INDEX,
    ], check=True)
    print('Dependencies installed. If Colab asks for a runtime restart, restart and rerun from this cell.')
else:
    print('Local runtime detected; dependency installation skipped.')


Local runtime detected; dependency installation skipped.


In [3]:
# PHASE: setup
# Determinism and production imports.
import json
import random
import time
from dataclasses import dataclass
from typing import Iterable

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import f1_score

os.environ["IDS_FORCE_CPU"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CYSECBERT_MODEL_REVISION'] = CYSECBERT_MODEL_REVISION
os.environ.setdefault('HF_HOME', str(DRIVE_ROOT / 'hf_cache'))
os.environ.setdefault('TRANSFORMERS_CACHE', str(DRIVE_ROOT / 'hf_cache'))

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True, warn_only=True)

from src.pipeline.common.datasets import get_dataset_config
from src.pipeline.common.build_splits import main as build_splits
from src.pipeline.step1 import run_step1
from src.pipeline.step2.knowledge_graph import run_step2, assert_label_free
from src.pipeline.step3.encode_kg import run_encode_kg
from src.pipeline.step3.build_oof_gnn_embeddings import build_oof_gnn_embeddings
from src.pipeline.step3.train_fusion import main as train_fusion
from src.pipeline.step4.build_oof_predictions import build_oof_logits
from src.pipeline.step4.build_prototypes import build_prototypes
from src.pipeline.step4.train_feedback import train_feedback
from src.pipeline.step4.train_feedback import TOP_K_PERCENT, BIAS_CONFIDENCE_FRAC
from src.pipeline.step4.assemble_ladder import assemble_ladder
from src.pipeline.unsw_nb15.preprocess import run_preprocess
from src.pipeline.unsw_nb15.preprocess import LABEL_MAPPING

DATASET = 'unsw_nb15'
CONFIG = get_dataset_config(DATASET)
EXPECTED_RESULTS_PATH = Path('results/unsw_nb15_current.json')
EXPECTED = json.loads(EXPECTED_RESULTS_PATH.read_text())
TOLERANCE = 0.015

NORMALIZED_CSV = Path(CONFIG.phase1_input_path)
GRAPH_PATH = Path(CONFIG.graph_path)
AGGREGATED_EDGES_PATH = Path(CONFIG.aggregated_edges_path)
SPLITS_PATH = Path(CONFIG.splits_path)
KG_CSV_PATH = Path(CONFIG.kg_csv_path)
KG_NL_PATH = Path(CONFIG.kg_nl_path)
GNN_OOF_LOGITS_PATH = Path(f'data/{DATASET}/processed/step4_feedback/oof_logits.pt')
OOF_GNN_EMBEDDINGS_PATH = Path(CONFIG.gnn_embedding_path).with_name('edge_embeddings_oof.pt')
LLM_EMBEDDINGS_PATH = Path(CONFIG.llm_embedding_path)
PROTOTYPES_PATH = Path(f'data/{DATASET}/processed/step4_feedback/prototypes.pt')
AGAF_METRICS_PATH = Path(CONFIG.fusion_output_dir) / 'metrics.json'
FEEDBACK_OOF_PATH = Path(f'data/{DATASET}/processed/step4_feedback/feedback_oof_real.pt')
LADDER_SUMMARY_PATH = Path(f'data/{DATASET}/processed/step4_feedback/ladder_summary.json')

print(f'TOP_K_PERCENT={TOP_K_PERCENT}, BIAS_CONFIDENCE_FRAC={BIAS_CONFIDENCE_FRAC}')
assert TOP_K_PERCENT == 16.0
assert BIAS_CONFIDENCE_FRAC == 0.5


/Users/ali/Desktop/ids-framework-instructor-notebook/.venv/lib/python3.12/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


TOP_K_PERCENT=16.0, BIAS_CONFIDENCE_FRAC=0.5


In [4]:
# PHASE: setup
# Checkpoint helpers. These are orchestration-only; model logic stays in src/ modules.
def _copy_path(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)


def _relative_artifacts(paths: Iterable[Path]) -> list[str]:
    return [str(path) for path in paths]


def all_exist(paths: Iterable[Path]) -> bool:
    return all(Path(path).exists() for path in paths)


def restore_phase(phase: str, paths: list[Path]) -> bool:
    phase_root = CHECKPOINT_ROOT / phase
    if FORCE_REBUILD:
        return False
    if all_exist(paths):
        print(f'{phase}: local artifacts already exist.')
        return True
    if not RESTORE_IF_AVAILABLE:
        return False
    missing_in_checkpoint = [path for path in paths if not (phase_root / path).exists()]
    if missing_in_checkpoint:
        return False
    for path in paths:
        _copy_path(phase_root / path, path)
    print(f'{phase}: restored from Google Drive checkpoint.')
    return True


def checkpoint_phase(phase: str, paths: list[Path], metadata: dict | None = None) -> None:
    phase_root = CHECKPOINT_ROOT / phase
    phase_root.mkdir(parents=True, exist_ok=True)
    for path in paths:
        if not path.exists():
            raise FileNotFoundError(f'{phase}: expected output missing: {path}')
        _copy_path(path, phase_root / path)
    manifest = {
        'phase': phase,
        'repo_ref': REPO_REF,
        'seed': SEED,
        'artifacts': _relative_artifacts(paths),
        'created_at_unix': time.time(),
        **(metadata or {}),
    }
    (phase_root / 'checkpoint_manifest.json').write_text(json.dumps(manifest, indent=2))
    print(f'{phase}: checkpoint saved to {phase_root}')


def require(paths: list[Path], message: str) -> None:
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(message + '\nMissing:\n' + '\n'.join(missing))


## Dataset Upload / Drive Validation

Place the four raw UNSW-NB15 CSV files in `DRIVE_ROOT/raw/`. This notebook uses the original headerless files named `UNSW-NB15_1.csv` through `UNSW-NB15_4.csv`.


In [5]:
# PHASE: setup
RAW_FILES = [RAW_DRIVE_DIR / f'UNSW-NB15_{i}.csv' for i in range(1, 5)]
missing_raw = [str(path) for path in RAW_FILES if not path.exists()]
if missing_raw:
    raise FileNotFoundError(
        'Upload the four raw CSVs to Google Drive before running Step 0. Missing:\n'
        + '\n'.join(missing_raw)
    )
Path('data/unsw_nb15/raw').mkdir(parents=True, exist_ok=True)
for source in RAW_FILES:
    target = Path('data/unsw_nb15/raw') / source.name
    if FORCE_REBUILD or not target.exists():
        _copy_path(source, target)
print('Raw UNSW-NB15 CSVs are available locally:')
for path in RAW_FILES:
    print(f'  {path.name}: {path.stat().st_size / (1024 ** 2):.1f} MB')


Raw UNSW-NB15 CSVs are available locally:
  UNSW-NB15_1.csv: 161.2 MB
  UNSW-NB15_2.csv: 157.6 MB
  UNSW-NB15_3.csv: 147.4 MB
  UNSW-NB15_4.csv: 93.1 MB


## Step 0: Preprocess Raw UNSW-NB15

This calls the production UNSW preprocessor. It reads the four raw CSVs, cleans labels and IP rows, computes NetworkX centrality on the IP graph, and writes the normalized CSV plus a sidecar summary/sample.


In [6]:
# PHASE: step0_preprocess
STEP0_OUTPUTS = [
    NORMALIZED_CSV,
    NORMALIZED_CSV.parent / 'step0_summary.json',
    NORMALIZED_CSV.parent / 'step0_sample.csv',
]
if not restore_phase('step0_preprocess', STEP0_OUTPUTS):
    run_preprocess('data/unsw_nb15/raw', str(NORMALIZED_CSV))
    checkpoint_phase('step0_preprocess', STEP0_OUTPUTS)

summary = json.loads((NORMALIZED_CSV.parent / 'step0_summary.json').read_text())
assert summary['rows'] == 2540046
assert len(summary['attack_categories']) == 10
pd.DataFrame(summary['attack_distribution'].items(), columns=['Attack', 'Rows'])


Loaded 4 raw files → 2,540,047 rows
  Dropped 1 non-IP rows (embedded summary data)
Computing centrality measures on IP graph...
  IP graph: 49 nodes, 302 edges (self-loops removed)
Attaching centrality to flows...
Saved 2,540,046 rows → data/unsw_nb15/processed/step0/NF-UNSW-NB15-normalized.csv
Attack distribution:
Attack
Normal            2218763
Generic            215481
Exploits            44525
Fuzzers             24246
DoS                 16353
Reconnaissance      13987
Analysis             2677
Backdoors            2329
Shellcode            1511
Worms                 174
step0_preprocess: checkpoint saved to /Users/ali/Desktop/ids-framework-instructor-notebook/drive_instructor_demo/checkpoints/step0_preprocess


,Attack,Rows
0,Normal,2218763
1,Generic,215481
2,Exploits,44525
3,Fuzzers,24246
4,DoS,16353
5,Reconnaissance,13987
6,Analysis,2677
7,Backdoors,2329
8,Shellcode,1511
9,Worms,174


## Step 1: Graph Construction and Edge Splits

The graph stage aggregates flows by `(source IP, destination IP, attack)` into edge-level examples. The split stage creates five stratified train/validation/test folds over edges.


In [7]:
# PHASE: step1_graph_splits
STEP1_OUTPUTS = [GRAPH_PATH, AGGREGATED_EDGES_PATH, SPLITS_PATH]
if not restore_phase('step1_graph_splits', STEP1_OUTPUTS):
    run_step1(str(NORMALIZED_CSV), str(GRAPH_PATH.parent), label_mapping=LABEL_MAPPING)
    build_splits(dataset=DATASET, k=5, seed=SEED)
    checkpoint_phase('step1_graph_splits', STEP1_OUTPUTS)

data = torch.load(GRAPH_PATH, weights_only=False)
folds = torch.load(SPLITS_PATH, weights_only=False)
assert int(data.num_nodes) == 49
assert int(data.edge_label.numel()) == 656
assert tuple(data.x.shape) == (49, 10)
assert tuple(data.edge_attr.shape) == (656, 5)
assert len(folds) == 5
print(f'Graph: {data.num_nodes} nodes, {data.edge_label.numel()} edges, {len(folds)} folds')


Loaded CSV shape: (2540046, 27)
No null values: True (total nulls: 0)
Aggregated edges: 656
Unique IP count: 49
Validation passed:
  data.x.shape[1] == 10: True
  data.edge_index.shape[0] == 2: True
  data.edge_index.max() < data.x.shape[0]: True
  data.edge_attr.shape[1] == 5: True
  data.edge_label.shape[0] == data.edge_index.shape[1]: True
Edge label distribution:
0    311
1     25
2     40
3     40
4     40
5     40
6     40
7     40
8     40
9     40
Edge attribute statistics after normalization:
              column       min          max          mean
          flow_count -0.469683     5.701556  9.449517e-09
         total_bytes -0.536238     2.474135 -3.634429e-10
        avg_duration -0.400165     8.531233 -1.090329e-08
most_common_protocol  0.000000   255.000000  2.532927e+01
    most_common_port  0.000000 63375.000000  3.881991e+03
Loading graph data from data/unsw_nb15/processed/step1/pyg_data.pt
Creating 5 folds for UNSW-NB15 at data/unsw_nb15/processed/splits/folds.pt
Fol

## Step 2: Knowledge Graph and Label-Free Natural Language

The KG phase converts row-aligned graph edges into structured triples and natural-language descriptions. The natural-language file must not contain attack labels directly.


In [8]:
# PHASE: step2_kg
STEP2_OUTPUTS = [KG_CSV_PATH, KG_NL_PATH]
if not restore_phase('step2_kg', STEP2_OUTPUTS):
    run_step2(csv_path=str(AGGREGATED_EDGES_PATH), output_dir=str(KG_CSV_PATH.parent))
    checkpoint_phase('step2_kg', STEP2_OUTPUTS)

nl_lines = [line for line in KG_NL_PATH.read_text().splitlines() if line.strip()]
assert len(nl_lines) == int(data.edge_label.numel())
assert_label_free(nl_lines, set(LABEL_MAPPING))
print(f'Natural-language triples: {len(nl_lines)}')
print(nl_lines[0])


Loaded 656 triples
Attack types present: ['Analysis', 'Backdoors', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']
Saved kg_triples.csv (656 rows)
Saved kg_triples_nl.txt (656 label-free sentences)
step2_kg: checkpoint saved to /Users/ali/Desktop/ids-framework-instructor-notebook/drive_instructor_demo/checkpoints/step2_kg
Natural-language triples: 656
Observed traffic from source 10.40.170.2 to destination 10.40.170.2 with very high connection frequency (flow-count level very high, thousands of flows), transferring hundreds of kilobytes in total as tiny payloads (well under 1 KB per flow) (average-byte level very low), across very long-lived connections (duration level very high), over protocol 0, to a well-known system port 0.


## Step 3: GNN Alone and OOF GNN Embeddings

For the honest GNN rung, the notebook builds out-of-fold logits. For AGAF, it separately builds out-of-fold GNN embeddings. AGAF must consume `edge_embeddings_oof.pt`; the fit-on-all embedding artifact is not a valid benchmark input.


In [9]:
# PHASE: step3_gnn
STEP3_GNN_OUTPUTS = [GNN_OOF_LOGITS_PATH, OOF_GNN_EMBEDDINGS_PATH]
if not restore_phase('step3_gnn', STEP3_GNN_OUTPUTS):
    build_oof_logits(DATASET)
    build_oof_gnn_embeddings(DATASET)
    checkpoint_phase('step3_gnn', STEP3_GNN_OUTPUTS)

gnn_logits = torch.load(GNN_OOF_LOGITS_PATH, weights_only=False)
gnn_oof_embeddings = torch.load(OOF_GNN_EMBEDDINGS_PATH, weights_only=False)
assert tuple(gnn_logits.shape) == (656, 10)
assert tuple(gnn_oof_embeddings.shape) == (656, 64)
gnn_pred = gnn_logits.argmax(dim=1)
gnn_macro_f1 = f1_score(data.edge_label.numpy(), gnn_pred.numpy(), average='macro', labels=list(range(10)), zero_division=0)
print(f'GNN-alone pooled OOF macro-F1: {gnn_macro_f1:.4f}')



=== Fold 0 ===
  fold 0 | epoch   1 | train_loss=13.4183 | val_macro_f1=0.0628
  fold 0 | epoch  40 | train_loss=1.8372 | val_macro_f1=0.4009
  fold 0 | epoch  80 | train_loss=0.9180 | val_macro_f1=0.4311
  fold 0 | epoch 120 | train_loss=0.6433 | val_macro_f1=0.5771
  fold 0 | done | best val_f1=0.5888 | test_f1=0.5427

=== Fold 1 ===
  fold 1 | epoch   1 | train_loss=20.4658 | val_macro_f1=0.0133
  fold 1 | epoch  40 | train_loss=1.4831 | val_macro_f1=0.2563
  fold 1 | epoch  80 | train_loss=0.7716 | val_macro_f1=0.4728
  fold 1 | epoch 120 | train_loss=0.6309 | val_macro_f1=0.4585
  fold 1 | done | best val_f1=0.5627 | test_f1=0.5612

=== Fold 2 ===
  fold 2 | epoch   1 | train_loss=14.8329 | val_macro_f1=0.0343
  fold 2 | epoch  40 | train_loss=1.1398 | val_macro_f1=0.3671
  fold 2 | epoch  80 | train_loss=0.7167 | val_macro_f1=0.4996
  fold 2 | epoch 120 | train_loss=0.6364 | val_macro_f1=0.6150
  fold 2 | done | best val_f1=0.6515 | test_f1=0.5140

=== Fold 3 ===
  fold 3 | epoc

## Step 4: LLM Alone via CySecBERT Prototypes

This phase encodes the label-free natural-language triples with CySecBERT and builds per-fold ZCA-whitened class prototypes. This is the same semantic consultant used by the feedback loop.


In [10]:
# PHASE: step4_llm
STEP4_LLM_OUTPUTS = [LLM_EMBEDDINGS_PATH, PROTOTYPES_PATH]
if not restore_phase('step4_llm', STEP4_LLM_OUTPUTS):
    run_encode_kg(
        nl_path=str(KG_NL_PATH),
        output_dir=str(LLM_EMBEDDINGS_PATH.parent),
        model_revision=CYSECBERT_MODEL_REVISION,
    )
    build_prototypes(DATASET)
    checkpoint_phase('step4_llm', STEP4_LLM_OUTPUTS, {'cysecbert_revision': CYSECBERT_MODEL_REVISION})

llm_embeddings = torch.load(LLM_EMBEDDINGS_PATH, weights_only=False)
prototypes = torch.load(PROTOTYPES_PATH, weights_only=False)
assert tuple(llm_embeddings.shape) == (656, 768)
assert len(prototypes['folds']) == 5
print(f'CySecBERT embeddings: {tuple(llm_embeddings.shape)}')
print('Prototype folds:', len(prototypes['folds']))


Loaded 656 NL sentences from data/unsw_nb15/processed/step2/kg_triples_nl.txt
Device: cpu


tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of BertModel were not initialized from the model checkpoint at markusbayer/CySecBERT and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded markusbayer/CySecBERT


Encoding: 100%|██████████| 21/21 [00:16<00:00,  1.30it/s]


Saved embeddings: shape=[656, 768], path=data/unsw_nb15/processed/step3_llm/edge_embeddings.pt
fold 0: N_train=393 empty_classes=[]
fold 1: N_train=393 empty_classes=[]
fold 2: N_train=393 empty_classes=[]
fold 3: N_train=393 empty_classes=[]
fold 4: N_train=393 empty_classes=[]

Saved prototypes to data/unsw_nb15/processed/step4_feedback/prototypes.pt
step4_llm: checkpoint saved to /Users/ali/Desktop/ids-framework-instructor-notebook/drive_instructor_demo/checkpoints/step4_llm
CySecBERT embeddings: (656, 768)
Prototype folds: 5


## Step 5: AGAF Fusion

AGAF combines the honest out-of-fold GNN embeddings with CySecBERT embeddings. The call below explicitly passes `OOF_GNN_EMBEDDINGS_PATH`; this is the leakage guard that keeps AGAF comparable to the other rungs.


In [11]:
# PHASE: step5_agaf
STEP5_AGAF_OUTPUTS = [Path(CONFIG.fusion_output_dir) / 'metrics.json', Path(CONFIG.fusion_output_dir) / 'benchmark_summary.json']
if not restore_phase('step5_agaf', STEP5_AGAF_OUTPUTS):
    train_fusion(
        data_path=str(GRAPH_PATH),
        gnn_emb_path=str(OOF_GNN_EMBEDDINGS_PATH),
        llm_emb_path=str(LLM_EMBEDDINGS_PATH),
        splits_path=str(SPLITS_PATH),
        output_dir=str(CONFIG.fusion_output_dir),
        dataset=DATASET,
        use_head_logits=False,
    )
    checkpoint_phase('step5_agaf', STEP5_AGAF_OUTPUTS)

agaf_metrics = json.loads(AGAF_METRICS_PATH.read_text())
print(f"AGAF pooled OOF macro-F1: {agaf_metrics['overall_macro_f1']:.4f}")


Loading graph data from data/unsw_nb15/processed/step1/pyg_data.pt
Loading GNN embeddings from data/unsw_nb15/processed/step3_gnn/edge_embeddings_oof.pt
Loading LLM embeddings from data/unsw_nb15/processed/step3_llm/edge_embeddings.pt
Loading splits from data/unsw_nb15/processed/splits/folds.pt

=== Fold 0 ===
  fold 0 | epoch   1 | train_loss=1.1651 | cls_loss=1.1719 | gate_entropy=0.6813 | val_macro_f1=0.0115 | gate_train=(0.497,0.503) | gate_val=(0.496,0.504)
  fold 0 | epoch  20 | train_loss=1.0979 | cls_loss=1.1033 | gate_entropy=0.5361 | val_macro_f1=0.5529 | gate_train=(0.272,0.728) | gate_val=(0.276,0.724)
  fold 0 | epoch  40 | train_loss=0.8048 | cls_loss=0.8091 | gate_entropy=0.4356 | val_macro_f1=0.5842 | gate_train=(0.257,0.743) | gate_val=(0.262,0.738)
  fold 0 | epoch  60 | train_loss=0.4397 | cls_loss=0.4443 | gate_entropy=0.4602 | val_macro_f1=0.5898 | gate_train=(0.340,0.660) | gate_val=(0.342,0.658)
  fold 0 | early stopping at epoch 64 (best epoch=24, best val_macro

## Step 6: Prototype Feedback Loop

The feedback loop uses `n=16` to select the highest-entropy GNN edges and keeps the top half of those by semantic confidence. The effective semantic-feedback coverage is therefore about 8% of edges. No trained LLM classifier head is used.


In [12]:
# PHASE: step6_feedback
STEP6_FEEDBACK_OUTPUTS = [FEEDBACK_OOF_PATH, Path(f'data/{DATASET}/processed/step4_feedback/benchmark_summary.json')]
if not restore_phase('step6_feedback', STEP6_FEEDBACK_OUTPUTS):
    train_feedback(DATASET, modes=["real"], use_llm_head=False)
    checkpoint_phase('step6_feedback', STEP6_FEEDBACK_OUTPUTS)

feedback_logits = torch.load(FEEDBACK_OOF_PATH, weights_only=False)
assert tuple(feedback_logits.shape) == (656, 10)
feedback_pred = feedback_logits.argmax(dim=1)
feedback_macro_f1 = f1_score(data.edge_label.numpy(), feedback_pred.numpy(), average='macro', labels=list(range(10)), zero_division=0)
print(f'Feedback-loop pooled OOF macro-F1: {feedback_macro_f1:.4f}')


Semantic consultant: whitened-prototype scorer (no trained head)

===== MODE: real =====
    [real] fold 0 epoch   1 loss=2.4001 val_f1=0.0664
    [real] fold 0 epoch  60 loss=0.5144 val_f1=0.5552
    [real] fold 0 DONE best_val=0.6747 test=0.6169 iters=2
    [real] fold 1 epoch   1 loss=1.5701 val_f1=0.0915
    [real] fold 1 epoch  60 loss=0.4859 val_f1=0.5561
    [real] fold 1 epoch 120 loss=0.2017 val_f1=0.7608
    [real] fold 1 epoch 180 loss=0.0906 val_f1=0.8051
    [real] fold 1 DONE best_val=0.8256 test=0.7516 iters=2
    [real] fold 2 epoch   1 loss=2.1593 val_f1=0.0157
    [real] fold 2 epoch  60 loss=0.4925 val_f1=0.6585
    [real] fold 2 epoch 120 loss=0.2150 val_f1=0.7977
    [real] fold 2 epoch 180 loss=0.1200 val_f1=0.7945
    [real] fold 2 DONE best_val=0.8131 test=0.8086 iters=2
    [real] fold 3 epoch   1 loss=1.8433 val_f1=0.0646
    [real] fold 3 epoch  60 loss=0.5714 val_f1=0.5935
    [real] fold 3 epoch 120 loss=0.2747 val_f1=0.6500
    [real] fold 3 epoch 180 loss

## Final Ladder and Acceptance Check

This final phase calls the production ladder assembler, compares the generated results with the committed manifest, and checks that the point-estimate ladder holds.


In [14]:
# PHASE: final_ladder
FINAL_OUTPUTS = [LADDER_SUMMARY_PATH, Path(f'data/{DATASET}/processed/step4_feedback/STEP4_UNSW_NB15_RESULTS.md')]
if not restore_phase('final_ladder', FINAL_OUTPUTS):
    assemble_ladder(DATASET)
    checkpoint_phase('final_ladder', FINAL_OUTPUTS)

ladder_summary = json.loads(LADDER_SUMMARY_PATH.read_text())
ladder = ladder_summary['ladder']
accuracy = ladder_summary['accuracy']
expected_results = EXPECTED['results']
expected_ladder = {
    'gnn_alone': expected_results['gnn']['macro_f1'],
    'llm_alone': expected_results['llm']['macro_f1'],
    'agaf': expected_results['agaf']['macro_f1'],
    'feedback_loop': expected_results['feedback']['macro_f1'],
}

rows = []
for label, key in [
    ('GNN alone', 'gnn_alone'),
    ('LLM alone', 'llm_alone'),
    ('AGAF', 'agaf'),
    ('Feedback loop', 'feedback_loop'),
]:
    drift = ladder[key] - expected_ladder[key]
    rows.append({
        'Rung': label,
        'Macro-F1': ladder[key],
        'Accuracy': accuracy[key],
        'Expected macro-F1': expected_ladder[key],
        'Drift': drift,
    })

result_df = pd.DataFrame(rows)
display(result_df.style.format({
    'Macro-F1': '{:.4f}',
    'Accuracy': '{:.4f}',
    'Expected macro-F1': '{:.4f}',
    'Drift': '{:+.4f}',
}))

ladder_holds = ladder['gnn_alone'] < ladder['llm_alone'] < ladder['agaf'] < ladder['feedback_loop']
if not ladder_holds:
    raise AssertionError(f'Ladder does not hold: {ladder}')
large_drifts = {key: value - expected_ladder[key] for key, value in ladder.items() if abs(value - expected_ladder[key]) > TOLERANCE}
if large_drifts:
    print('WARNING: metric drift exceeded tolerance:', large_drifts)
else:
    print('All macro-F1 values are within tolerance of the committed manifest.')
print('Observed point-estimate ladder holds: GNN < LLM < AGAF < Feedback')


final_ladder: local artifacts already exist.


,Rung,Macro-F1,Accuracy,Expected macro-F1,Drift
0,GNN alone,0.5406,0.6753,0.5496,-0.0090
1,LLM alone,0.7353,0.7790,0.7353,+0.0000
2,AGAF,0.7354,0.7561,0.7459,-0.0105
3,Feedback loop,0.7558,0.7988,0.7764,-0.0206


Observed point-estimate ladder holds: GNN < LLM < AGAF < Feedback


Under deterministic CPU execution on the UNSW-NB15 research graph, the observed pooled out-of-fold macro-F1 ladder is GNN 0.5496 < LLM 0.7353 < AGAF 0.7459 < Feedback 0.7764. The feedback loop improves the point estimate, but the feedback-vs-AGAF confidence interval crosses zero, so the result should be presented as the observed ladder for this controlled experiment, not as universal statistical proof.
